# End-to-End Chatbot Workflow

This notebook brings together location lookup, weather retrieval, activity-based assessment, and language-model response generation.

It demonstrates how a user question moves through the main processing steps and how the resulting forecast information is returned as a conversational answer.


### Shared project OpenAI client
Run `00_Project_Setup.ipynb` once first. This notebook then uses the shared project configuration; no API key is entered here.

In [21]:
import os
import json
import requests
import pandas as pd

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


### Project-level LLM client
This notebook uses the API configuration initialized in `00_Project_Setup.ipynb`. No API key is entered here.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


## 1. Install required packages

In [3]:
%pip -q install -U openai requests pandas

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and secure API key

Store your new API key in Google Colab Secrets with the name:

`OPENAI_API_KEY`

Do not put the actual key into this notebook.

# PART A — Weather Tool
## 3. Location geocoding

In [4]:
def geocode_location(location):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": location,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    if "results" not in data or not data["results"]:
        raise ValueError(f"Location not found: {location}")

    result = data["results"][0]

    return {
        "name": result["name"],
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone", "auto")
    }

## 4. Open-Meteo forecast function

In [5]:
def get_weather_forecast(
    latitude,
    longitude,
    timezone="auto",
    forecast_days=3
):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join([
            "temperature_2m",
            "precipitation_probability",
            "precipitation",
            "weather_code",
            "cloud_cover",
            "wind_speed_10m"
        ]),
        "timezone": timezone,
        "forecast_days": forecast_days,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()

In [6]:
WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail"
}

def weather_code_to_description(code):
    return WEATHER_CODES.get(int(code), "Unknown weather condition")

# PART B — Activity Weather Logic
## 5. Activity profiles

These are prototype heuristics for the prototype. They are not safety guarantees.

In [7]:
ACTIVITY_PROFILES = {
    "hiking": {"max_rain_probability": 60, "max_wind_kmh": 35, "low_temp_c": 0, "high_temp_c": 30},
    "cycling": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 2, "high_temp_c": 32},
    "mountain biking": {"max_rain_probability": 45, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 28},
    "running": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 30},
    "trail running": {"max_rain_probability": 45, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 28},
    "camping": {"max_rain_probability": 50, "max_wind_kmh": 25, "low_temp_c": 5, "high_temp_c": 30},
    "outdoor tour": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 2, "high_temp_c": 32}
}

In [8]:
def assess_activity_weather(
    activity,
    temperature,
    wind_speed,
    precipitation_probability,
    weather_description=None
):
    activity_key = activity.lower().strip()

    if activity_key not in ACTIVITY_PROFILES:
        raise ValueError(f"Unsupported activity: {activity}")

    profile = ACTIVITY_PROFILES[activity_key]
    concerns = []

    if temperature < profile["low_temp_c"]:
        concerns.append("low temperature")
    if temperature > profile["high_temp_c"]:
        concerns.append("high temperature")
    if wind_speed > profile["max_wind_kmh"]:
        concerns.append("strong wind")
    if precipitation_probability > profile["max_rain_probability"]:
        concerns.append("high precipitation probability")

    summary = (
        "Main factors: " + ", ".join(concerns)
        if concerns
        else "No major weather concern identified by the prototype rules."
    )

    return {
        "activity": activity_key,
        "temperature_c": temperature,
        "wind_kmh": wind_speed,
        "precipitation_probability": precipitation_probability,
        "weather_description": weather_description,
        "concerns": concerns,
        "summary": summary
    }

# PART C — LLM Weather Agent
## 6. Structured request schema

In [9]:
REQUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "activity": {
            "type": "string",
            "description": "Outdoor activity requested by the user."
        },
        "location": {
            "type": "string",
            "description": "Place for which weather is requested."
        },
        "date": {
            "type": "string",
            "description": "Requested date as stated by the user."
        },
        "time_period": {
            "type": "string",
            "description": "Morning, afternoon, evening, overnight, specific time, or empty."
        },
        "elevation_m": {
            "type": ["number", "null"],
            "description": "Requested elevation in metres, if explicitly provided."
        }
    },
    "required": ["activity", "location", "date", "time_period", "elevation_m"],
    "additionalProperties": False
}

In [10]:
SYSTEM_PROMPT = """
You are the Weather Forecast Chatbot's request-understanding agent.

Extract:
- activity
- location
- date
- time period
- elevation in metres when explicitly provided

Supported activities:
hiking, cycling, mountain biking, running, camping,
trail running, outdoor tour.

Rules:
1. Extract only information supported by the user's wording.
2. Do not invent missing values.
3. Preserve relative date wording such as "tomorrow" or "Saturday".
4. If no time period is given, return an empty string.
5. If no elevation is given, return null.
6. Return only the structured data requested.
"""

In [11]:
def extract_weather_request(user_question):
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=SYSTEM_PROMPT,
        input=user_question,
        text={
            "format": {
                "type": "json_schema",
                "name": "weather_request",
                "schema": REQUEST_SCHEMA,
                "strict": True
            }
        }
    )
    return json.loads(response.output_text)

# PART D — Forecast Selection
## 7. Select the requested forecast period

This prototype supports `today`, `tomorrow`, `morning`, `afternoon`, `evening`, and `overnight`.

The date/time resolver can be expanded later for all natural-language date expressions.

In [12]:
def select_forecast_period(forecast_df, date_text, time_period):
    df = forecast_df.copy()
    df["datetime"] = pd.to_datetime(df["time"])

    today = pd.Timestamp.now(tz=None).normalize()
    tomorrow = today + pd.Timedelta(days=1)

    date_lower = date_text.lower().strip()

    if date_lower == "tomorrow":
        target_date = tomorrow.date()
    elif date_lower == "today":
        target_date = today.date()
    else:
        # POC fallback for dates not yet handled by the resolver.
        target_date = df["datetime"].dt.date.iloc[0]

    df = df[df["datetime"].dt.date == target_date]

    period = time_period.lower().strip()

    if period == "morning":
        df = df[df["datetime"].dt.hour.between(6, 11)]
    elif period == "afternoon":
        df = df[df["datetime"].dt.hour.between(12, 17)]
    elif period == "evening":
        df = df[df["datetime"].dt.hour.between(18, 21)]
    elif period == "overnight":
        df = df[(df["datetime"].dt.hour >= 22) | (df["datetime"].dt.hour <= 5)]

    return df

# PART E — End-to-End Pipeline
## 8. Retrieve forecast for the extracted request

In [13]:
def retrieve_forecast_for_request(request):
    location = geocode_location(request["location"])

    weather_data = get_weather_forecast(
        latitude=location["latitude"],
        longitude=location["longitude"],
        timezone=location["timezone"],
        forecast_days=3
    )

    forecast_df = pd.DataFrame(weather_data["hourly"])

    selected_df = select_forecast_period(
        forecast_df,
        request["date"],
        request["time_period"]
    )

    return location, weather_data, selected_df

## 9. Aggregate the selected forecast

In [14]:
def summarize_forecast(selected_df):
    if selected_df.empty:
        return None

    weather_codes = selected_df["weather_code"].astype(int)

    return {
        "temperature_min_c": round(float(selected_df["temperature_2m"].min()), 1),
        "temperature_max_c": round(float(selected_df["temperature_2m"].max()), 1),
        "wind_max_kmh": round(float(selected_df["wind_speed_10m"].max()), 1),
        "precipitation_probability_max": round(
            float(selected_df["precipitation_probability"].max()), 1
        ),
        "precipitation_total_mm": round(
            float(selected_df["precipitation"].sum()), 1
        ),
        "conditions": list(dict.fromkeys(
            weather_code_to_description(code)
            for code in weather_codes
        ))
    }

## 10. Generate the final chatbot response

The LLM receives the factual forecast and structured activity assessment. It is instructed not to invent weather values.

In [15]:
def generate_weather_response(
    user_question,
    request,
    location,
    forecast_summary,
    activity_assessment
):
    response_prompt = f"""
User question:
{user_question}

Structured request:
{json.dumps(request, ensure_ascii=False)}

Location:
{json.dumps(location, ensure_ascii=False)}

Forecast data:
{json.dumps(forecast_summary, ensure_ascii=False)}

Activity assessment:
{json.dumps(activity_assessment, ensure_ascii=False)}

Write a concise answer for the outdoor user.

Requirements:
- Use only the supplied forecast data for factual weather values.
- Mention temperature, conditions and wind.
- Mention precipitation when relevant.
- Explain the main activity-specific weather factor.
- Do not claim that conditions are guaranteed safe.
- Do not invent missing information.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        instructions=(
            "You are the response-generation component of a weather "
            "forecast chatbot. Be concise, factual and transparent."
        ),
        input=response_prompt
    )

    return response.output_text

# 11. First complete prototype query

We use the example scenario:

> **“Will it rain during my cycling tour in Berlin tomorrow afternoon?”**

In [22]:
user_question = (
    "Will it rain during my cycling tour in Berlin tomorrow afternoon?"
)

print("USER:")
print(user_question)

# 1. LLM understands the request
request = extract_weather_request(user_question)

print("\nEXTRACTED REQUEST:")
print(request)

# 2. Weather Tool retrieves factual data
location, weather_data, selected_forecast = retrieve_forecast_for_request(request)

print("\nLOCATION:")
print(location)

# 3. Summarize the selected forecast
forecast_summary = summarize_forecast(selected_forecast)

if forecast_summary is None:
    raise ValueError("No forecast records were found for the requested period.")

print("\nFORECAST SUMMARY:")
print(forecast_summary)

# 4. Activity-specific interpretation
average_temperature = (
    forecast_summary["temperature_min_c"]
    + forecast_summary["temperature_max_c"]
) / 2

activity_assessment = assess_activity_weather(
    activity=request["activity"],
    temperature=average_temperature,
    wind_speed=forecast_summary["wind_max_kmh"],
    precipitation_probability=forecast_summary["precipitation_probability_max"],
    weather_description=", ".join(forecast_summary["conditions"])
)

print("\nACTIVITY ASSESSMENT:")
print(activity_assessment)

# 5. Generate the final answer
final_answer = generate_weather_response(
    user_question=user_question,
    request=request,
    location=location,
    forecast_summary=forecast_summary,
    activity_assessment=activity_assessment
)

print("\nCHATBOT RESPONSE:")
print(final_answer)

USER:
Will it rain during my cycling tour in Berlin tomorrow afternoon?

EXTRACTED REQUEST:
{'activity': 'cycling', 'location': 'Berlin', 'date': 'tomorrow', 'time_period': 'afternoon', 'elevation_m': None}

LOCATION:
{'name': 'Berlin', 'latitude': 52.52437, 'longitude': 13.41053, 'country': 'Germany', 'timezone': 'Europe/Berlin'}

FORECAST SUMMARY:
{'temperature_min_c': 12.8, 'temperature_max_c': 16.6, 'wind_max_kmh': 10.4, 'precipitation_probability_max': 0.0, 'precipitation_total_mm': 0.0, 'conditions': ['Overcast']}

ACTIVITY ASSESSMENT:
{'activity': 'cycling', 'temperature_c': 14.700000000000001, 'wind_kmh': 10.4, 'precipitation_probability': 0.0, 'weather_description': 'Overcast', 'concerns': [], 'summary': 'No major weather concern identified by the prototype rules.'}

CHATBOT RESPONSE:
Short answer: No — no rain is expected for your cycling tour in Berlin tomorrow afternoon (precipitation probability 0%, 0.0 mm).

Details: temperature about 14.7 °C, sky overcast, winds around 1

# 12. Reusable chatbot function

After the first successful test, this function allows us to send other natural-language questions through the same complete pipeline.

In [23]:
def run_chatbot(user_question):
    request = extract_weather_request(user_question)

    location, weather_data, selected_forecast = retrieve_forecast_for_request(
        request
    )

    forecast_summary = summarize_forecast(selected_forecast)

    if forecast_summary is None:
        return {
            "request": request,
            "error": "No forecast records found for the requested period."
        }

    average_temperature = (
        forecast_summary["temperature_min_c"]
        + forecast_summary["temperature_max_c"]
    ) / 2

    activity_assessment = assess_activity_weather(
        activity=request["activity"],
        temperature=average_temperature,
        wind_speed=forecast_summary["wind_max_kmh"],
        precipitation_probability=forecast_summary["precipitation_probability_max"],
        weather_description=", ".join(forecast_summary["conditions"])
    )

    answer = generate_weather_response(
        user_question,
        request,
        location,
        forecast_summary,
        activity_assessment
    )

    return {
        "request": request,
        "location": location,
        "forecast": forecast_summary,
        "assessment": activity_assessment,
        "answer": answer
    }

# 13. Test a second example scenario

This tests the Munich hiking scenario.

In [24]:
result = run_chatbot(
    "What's the weather like for my hike in Munich tomorrow?"
)

print(result["answer"])

Tomorrow in Munich: temperatures 5.8 °C (low) to 18.3 °C (high) — average ~12.1 °C. Conditions: clear to mainly clear with some partly cloudy spells. Wind: light, max ~6.2 km/h. Precipitation: none expected (0 mm, 0% probability).

Main hiking factor: the temperature range — cool at the low end (early/late) and mild by daytime — so dress in layers and plan for cooler starts/finishes. No major weather concern identified, but conditions are not guaranteed safe.


# 14. End-to-end architecture summary

```text
                    USER
                      │
                      ▼
             ┌─────────────────┐
             │   LLM AGENT     │
             │ Request         │
             │ extraction      │
             └────────┬────────┘
                      │
                structured data
                      │
                      ▼
             ┌─────────────────┐
             │  WEATHER TOOL   │
             │ Geocoding + API │
             └────────┬────────┘
                      │
                      ▼
             ┌─────────────────┐
             │   OPEN-METEO    │
             │   Forecast API  │
             └────────┬────────┘
                      │
                 factual data
                      │
                      ▼
             ┌─────────────────┐
             │ ACTIVITY LOGIC  │
             │ Interpretation  │
             └────────┬────────┘
                      │
                  assessment
                      │
                      ▼
             ┌─────────────────┐
             │  LLM RESPONSE   │
             │   GENERATION    │
             └────────┬────────┘
                      │
                      ▼
                    USER
```

### Current POC status

- [x] Location geocoding
- [x] Open-Meteo forecast retrieval
- [x] Required weather variables
- [x] Activity-specific interpretation
- [x] LLM natural-language extraction
- [x] Structured data flow
- [x] LLM response generation
- [x] End-to-end test

### Next development step

After this notebook runs successfully, we should:

1. improve date/time handling;
2. test all 10 example scenarios;
3. add the Streamlit interface;
4. record test results and limitations;
5. prepare the approximately 10-slide project documentation.